In [ ]:
#====================================================================================
# Main business service to entity
#====================================================================================


import pandas as pd

# ==========================================
# 1. Paths and keywords
# ==========================================
from pathlib import Path
ROOT = Path("path/to/project")
file_input =  ROOT / "data" / "raw"   / "主营业务.xls"
file_output = ROOT / "data" / "clean" / "主营业务_服务业里加制造业.xlsx"

# Service keywords (used to identify the initial type of business)
service_keywords = [
    '研发', '技术转让', '技术咨询', '合同能源管理', '工程勘察', '工程勘探', '软件', 
    '电路设计', '电路测试', '信息系统', '业务流程管理', '信息技术', '设计', '商标', 
    '著作权', '知识产权', '专利', '广告', '会议展览', '会展', '文化创意', '航空地面服务', 
    '航空服务', '港口', '码头', '场站', '票务', '打捞', '救助', '货代', '货物运输代理', 
    '代理报关', '仓储', '装卸', '搬运', '收派', '物流辅助', '认证服务', '鉴证服务', 
    '财务咨询', '税收咨询', '法律咨询', '管理咨询', '业务运作', '流程管理', '鉴证咨询', 
    '广播', '影视', '运输', '货运', '邮政', '电信'
]

# Real-economy keywords (covering sections A, B, C, D and E)
entity_keywords = [
    # Agriculture, forestry, animal husbandry and fishery
    '农业', '林业', '畜牧', '渔业', '种植', '养殖', '采摘',
    # Mining
    '采矿', '采选', '选矿', '开采', '矿产', '煤炭', '石油', '天然气', '铁矿', '金矿', '铜矿',
    # Manufacturing: raw materials
    '冶炼', '压延', '铸造', '锻造', '化工', '炼焦', '精炼', '橡胶', '塑料', '非金属矿物',
    '水泥', '玻璃', '陶瓷', '砖瓦', '耐火材料', '木材', '人造板', '纸浆', '造纸',
    # Manufacturing: consumer goods and processing
    '加工', '纺织', '印染', '织造', '服装', '服饰', '皮革', '毛皮', '羽绒', '家具',
    '食品', '饮料', '烟草', '罐头', '调味品', '屠宰', '医药制造', '化学纤维',
    # Manufacturing: equipment and technology
    '制造', '组装', '金属制品', '通用设备', '专用设备', '机械', '机床', '锅炉', '泵', '阀门',
    '轴承', '齿轮', '电梯', '风机', '汽车', '发动机', '零部件', '摩托车', '自行车',
    '铁路设备', '船舶', '航空器', '航天', '雷达', '电子设备', '仪器仪表', '办公机械',
    # Energy and utilities
    '电力', '热力', '燃气', '自来水', '发电', '输电', '变电', '配电', '核能', '太阳能', '风能',
    # Construction and engineering
    '建筑', '土木', '工程施工', '公路建设', '桥梁', '隧道', '河道', '港口建设', '装饰', '装修',
    '安装工程', '房地产开发', '房产开发'
]

service_pattern = '|'.join(service_keywords)
entity_pattern = '|'.join(entity_keywords)

# ==========================================
# 2. Data processing
# ==========================================
print("Reading data...")
df = pd.read_excel(file_input)
df.columns = df.columns.str.strip()

# Sort so that shift() works along each firm's time series
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['code_str'] = df['code_str'].astype(str)
df = df.sort_values(by=['code_str', 'year'])

# ==========================================
# 3. Identification: service firm -> adds real-economy business
# ==========================================
print("Identifying changes in main business...")

# Keywords in the current year
df['has_service_kw'] = df['MAINBUSSINESS_11'].str.contains(service_pattern, case=False, na=False).astype(int)
df['has_entity_kw'] = df['MAINBUSSINESS_11'].str.contains(entity_pattern, case=False, na=False).astype(int)

# Keywords in the previous observed year
df['prev_service_kw'] = df.groupby('code_str')['has_service_kw'].shift(1)
df['prev_entity_kw'] = df.groupby('code_str')['has_entity_kw'].shift(1)

# Change indicator (scope_change_bin)
# Conditions: 1. last year was a pure service firm (service keywords, no real-economy keywords)
#             2. real-economy keywords appear this year
df['scope_change_bin'] = (
    (df['prev_service_kw'] == 1) & 
    (df['prev_entity_kw'] == 0) & 
    (df['has_entity_kw'] == 1)
).astype(int)

# ==========================================
# 4. Export
# ==========================================
df.to_excel(file_output, index=False)
print("-" * 30)
print(f"Done. Number of firm-years with a change: {df['scope_change_bin'].sum()}")
print(f"Results saved to: {file_output}")

In [ ]:
#====================================================================================
# Main business add service
#====================================================================================


import pandas as pd
import re

# ==========================================
# 1. Paths and settings
# ==========================================
file_input =  ROOT / "data" / "raw"   / "主营业务.xls"
file_output = ROOT / "data" / "clean" / "主营业务_制造业里加服务业.xlsx"

# Keyword list
keywords = [
    '研发', '技术转让', '技术咨询', '合同能源管理', '工程勘察', '工程勘探', '软件', 
    '电路设计', '电路测试', '信息系统', '业务流程管理', '信息技术', '设计', '商标', 
    '著作权', '知识产权', '专利', '广告', '会议展览', '会展', '文化创意', '航空地面服务', 
    '航空服务', '港口', '码头', '场站', '票务', '打捞', '救助', '货代', '货物运输代理', 
    '代理报关', '仓储', '装卸', '搬运', '收派', '物流辅助', '认证服务', '鉴证服务', 
    '财务咨询', '税收咨询', '法律咨询', '管理咨询', '业务运作', '流程管理', '鉴证咨询', 
    '广播', '影视', '运输', '货运', '邮政', '电信'
]

# Build the regular-expression pattern
pattern = '|'.join(keywords)

# ==========================================
# 2. Read and preprocess
# ==========================================
print("Reading data, please wait...")
# For large files, saving as .csv first makes reading faster
df = pd.read_excel(file_input)

# Remove spaces from column names
df.columns = df.columns.str.strip()

# Make sure code_str and year have the correct format
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['code_str'] = df['code_str'].astype(str)

# Sort by firm and year (required to identify changes)
df = df.sort_values(by=['code_str', 'year'])

# ==========================================
# 3. Identification
# ==========================================
print("Matching keywords and flagging changes...")

# A. Whether the current year contains a keyword (1 = yes, 0 = no)
# na=False avoids errors on missing values
df['has_service_kw'] = df['MAINBUSSINESS_11'].str.contains(pattern, case=False, na=False).astype(int)

# B. Keyword status in the previous year (using shift)
df['prev_service_kw'] = df.groupby('code_str')['has_service_kw'].shift(1)

# C. Change indicator: scope_change_bin
# Logic: no keyword last year (0) and a keyword this year (1) -> 1 (service business added)
df['scope_change_bin'] = (
    (df['has_service_kw'] == 1) & (df['prev_service_kw'] == 0)
).astype(int)

# ==========================================
# 4. Summary and export
# ==========================================
# Number of firm-years with a change
total_changes = df['scope_change_bin'].sum()
# Number of firms involved
total_firms = df[df['scope_change_bin'] == 1]['code_str'].nunique()

print("-" * 30)
print(f"Identification complete.")
print(f"Firm-years with a change in main business: {total_changes}")
print(f"Number of listed firms involved: {total_firms}")
print("-" * 30)

# Export results
print(f"Exporting results to: {file_output}")
df.to_excel(file_output, index=False)
print("Export complete. See the scope_change_bin column in Excel.")